# CORA-ML (Basic Model)
### 2-layer GCN

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv
from torch_geometric.utils import train_test_split_edges
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

The following is implemented using Danish's GCN model which he sent us earlier in the quarter.

In [20]:
# Define the GCN model
class GCN(nn.Module):
    def __init__(self, in_feats, h_feats, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_feats, h_feats)
        self.conv2 = GCNConv(h_feats, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        return h

# Load the datasets
cora_dataset = CitationFull(root='/tmp/CoraML', name='Cora_ML')
data = cora_dataset[0]

num_nodes = data.num_nodes
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# Assign masks (this is just an example of random splitting)
indices = torch.randperm(num_nodes)
train_indices = indices[:int(0.2 * num_nodes)]
test_indices = indices[int(0.8 * num_nodes):]

train_mask[train_indices] = True
test_mask[test_indices] = True

data.train_mask = train_mask
data.test_mask = test_mask

data = data.to(device)

print(data)
# print(cora_dataset[0])
# data = cora_dataset[0]
# data = data.to(device)

Data(x=[2995, 2879], edge_index=[2, 16316], y=[2995], train_mask=[2995], test_mask=[2995])


In [21]:
in_feats = data.x.shape[1]
h_feats = 64
num_classes = cora_dataset.num_classes
model = GCN(in_feats, h_feats, num_classes)
model = model.to(device)

def train(model, data, train_mask, labels):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    logits = model(data.cuda())
    
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

### Uncomment the following cell if you want to save embeddings (Wyatt will likely do this, not sure - not my topic)

In [4]:
# def save_embeddings_and_edges(dataset, dataset_name):
#     model.eval()
#     with torch.no_grad():
#         embeddings = model(g, features)
#     print(embeddings.shape)
#     np.save(f'../embeddings/{dataset_name}_embeddings.npy', embeddings.detach().numpy())


#     edge_index = g.edges()
#     np.save(f'../embeddings/{dataset_name}_edge_index.npy', np.vstack((edge_index[0].numpy(), edge_index[1].numpy())))

In [22]:
train(model, data, data.train_mask, data.y)
# save_embeddings_and_edges(cora_dataset, 'cora')

1.9312092065811157

In [23]:
def test():
    # data = cora_dataset[0].to(device)
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    # incorrect_indices = (pred[data.test_mask] != data.y[data.test_mask]).nonzero()

    # print("Incorrect Predictions Indices:", incorrect_indices.flatten().tolist())

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.5906, Accuracy: 0.6678
Epoch: 001, Loss: 1.2633, Accuracy: 0.5008
Epoch: 002, Loss: 0.9714, Accuracy: 0.6361
Epoch: 003, Loss: 1.0990, Accuracy: 0.6377
Epoch: 004, Loss: 0.7345, Accuracy: 0.6594
Epoch: 005, Loss: 0.9017, Accuracy: 0.6962
Epoch: 006, Loss: 0.5899, Accuracy: 0.6678
Epoch: 007, Loss: 0.7786, Accuracy: 0.7212
Epoch: 008, Loss: 0.4861, Accuracy: 0.6895
Epoch: 009, Loss: 0.6832, Accuracy: 0.7396
Epoch: 010, Loss: 0.4190, Accuracy: 0.6912
Epoch: 011, Loss: 0.6118, Accuracy: 0.7596
Epoch: 012, Loss: 0.3765, Accuracy: 0.6828
Epoch: 013, Loss: 0.5542, Accuracy: 0.7696
Epoch: 014, Loss: 0.3420, Accuracy: 0.6978
Epoch: 015, Loss: 0.5043, Accuracy: 0.7713
Epoch: 016, Loss: 0.3157, Accuracy: 0.7012
Epoch: 017, Loss: 0.4643, Accuracy: 0.7713
Epoch: 018, Loss: 0.2946, Accuracy: 0.7145
Epoch: 019, Loss: 0.4303, Accuracy: 0.7763
Epoch: 020, Loss: 0.2758, Accuracy: 0.7212
Epoch: 021, Loss: 0.4010, Accuracy: 0.7796
Epoch: 022, Loss: 0.2598, Accuracy: 0.7245
Epoch: 023,

In [24]:
torch.save(model.state_dict(), 'cora_ml_gcn.pt')

The following shows the 10 nodes with the highest degree.

In [8]:
# import networkx as nx
# from dgl import to_networkx
# import matplotlib.pyplot as plt

# G = to_networkx(g)
# pos = nx.spring_layout(G, seed=42)
# cent = nx.degree_centrality(G)
# node_size = list(map(lambda x: x * 500, cent.values()))
# cent_array = np.array(list(cent.values()))
# threshold = sorted(cent_array, reverse=True)[50]
# print("threshold", threshold)
# cent_bin = np.where(cent_array >= threshold, 1, 0.1)
# plt.figure(figsize=(12, 12))
# nodes = nx.draw_networkx_nodes(G, pos, node_size=node_size,
#                                cmap=plt.cm.plasma,
#                                node_color=cent_bin,
#                                nodelist=list(cent.keys()),
#                                alpha=cent_bin)
# edges = nx.draw_networkx_edges(G, pos, width=0.25, alpha=0.3)
# plt.show()

This shows me the node with the highest degree.

In [9]:
# threshold = sorted(cent_array, reverse=True)[0]
# print("threshold", threshold)
# cent_bin = np.where(cent_array >= threshold, 1, 0.1)
# plt.figure(figsize=(12, 12))
# nodes = nx.draw_networkx_nodes(G, pos, node_size=node_size,
#                                cmap=plt.cm.plasma,
#                                node_color=cent_bin,
#                                nodelist=list(cent.keys()),
#                                alpha=cent_bin)
# edges = nx.draw_networkx_edges(G, pos, width=0.25, alpha=0.3)
# plt.show()